# Topic 21 — NLP Fundamentals: Text Preprocessing
### ⭐ Your serious NLP path starts here. Theory → tiny example → nltk → spaCy → a full preprocessing function.

Text is unstructured — before any ML algorithm can use it, you must turn it into a clean, consistent
form. Every step below is a choice, not a mandatory rule.

> **Important for cyberbullying/toxicity detection specifically**: don't blindly strip everything.
> Hashtags, punctuation, emojis, repeated characters ("stoooopid"), and CAPITALIZATION often carry
> real signal about aggression/intensity — decide deliberately what to keep.

In [ ]:
import re
import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("averaged_perceptron_tagger", quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

!python -m spacy download en_core_web_sm -q
import spacy
nlp = spacy.load("en_core_web_sm")

## 1. Lowercasing

Simple but consequential: makes "Free" and "free" the same token. Usually helps, but note it also
throws away ALL-CAPS as a signal (which, per the note above, can matter for toxicity detection).

In [ ]:
text = "You are SO Stupid!!! Nobody Likes You :("
print("lowercased:", text.lower())

## 2. Sentence and word tokenization

**Tokenization** = splitting text into pieces (tokens). Sentence tokenization splits into
sentences; word tokenization splits into words/punctuation marks.

In [ ]:
paragraph = "You are stupid. Nobody likes you! Just leave, please."

sentences = sent_tokenize(paragraph)
print("sentences:", sentences)

words = word_tokenize(paragraph)
print("words:", words)
# Notice tokenization splits off punctuation as its own tokens ('.', '!', ',').

## 3. Punctuation, stopwords

- **Punctuation**: often removed, but `!` and `?` can signal emotional intensity in toxic text.
- **Stopwords**: very common words ("the", "is", "a") that carry little topical meaning on their own —
  often removed for topic modeling, but sometimes kept for tasks sensitive to phrasing/tone.

In [ ]:
stop_words = set(stopwords.words("english"))
print("sample stopwords:", list(stop_words)[:15])

tokens = word_tokenize("you are such a stupid and worthless person")
no_stopwords = [t for t in tokens if t.lower() not in stop_words]
print("original:", tokens)
print("without stopwords:", no_stopwords)
# Notice: removing stopwords here keeps the emotionally loaded words (stupid, worthless) intact
# while dropping mostly grammatical scaffolding.

## 4. Stemming vs Lemmatization

Both reduce words to a base form, but differently:
- **Stemming**: crude, rule-based chopping of word endings. Fast, but can produce non-words
  (e.g. "studies" -> "studi").
- **Lemmatization**: uses vocabulary/grammar rules to return a real base word (the "lemma"),
  e.g. "studies" -> "study", "worse" -> "bad". Slower, more accurate.

In [ ]:
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

words_to_test = ["studies", "studying", "worthless", "running", "better", "worse"]

print(f"{'word':<12}{'stemmed':<12}{'lemmatized':<12}")
for w in words_to_test:
    print(f"{w:<12}{stemmer.stem(w):<12}{lemmatizer.lemmatize(w):<12}")
# Notice stemming's "studi" isn't a real word, but lemmatization's "study" is.
# For "better"/"worse" you'd need pos='a' (adjective) for correct lemmatization -- see next cell.

In [ ]:
# Lemmatization is more accurate when you tell it the part of speech
print(lemmatizer.lemmatize("worse", pos="a"))    # 'a' = adjective -> "bad"
print(lemmatizer.lemmatize("running", pos="v"))   # 'v' = verb -> "run"
# Rule of thumb: for social-media text classification, stemming is often "good enough" and much
# faster; use lemmatization when word meaning precision matters more than speed.

## 5. Social-media specific noise: URLs, mentions, hashtags, emojis, numbers

In [ ]:
social_text = "check this out http://example.com @john #bullying123 you're worthless 😢 call me at 12345"

# URLs
no_urls = re.sub(r"http\S+|www\.\S+", "", social_text)
print("no urls:", no_urls)

# Mentions (@username)
no_mentions = re.sub(r"@\w+", "", no_urls)
print("no mentions:", no_mentions)

# Hashtags -- often better to KEEP the word but drop the '#' symbol, not delete entirely
hashtag_word_kept = re.sub(r"#(\w+)", r"\1", no_mentions)
print("hashtags -> plain words:", hashtag_word_kept)

# Numbers
no_numbers = re.sub(r"\d+", "", hashtag_word_kept)
print("no numbers:", no_numbers)

# Emojis -- decide deliberately: remove, or convert to a text label (often more useful for toxicity!)
import unicodedata
def has_emoji(s):
    return any(unicodedata.category(ch) == "So" for ch in s)
print("contains emoji:", has_emoji(social_text))
# A sentiment/emoji library like `emoji` can convert 😢 -> ":crying_face:" -- often MORE useful
# than deleting it, since emojis carry real emotional signal for toxicity detection.

## 6. Contractions & repeated characters

In [ ]:
contracted = "i can't believe you're doing this, you won't get away with it"

# Simple contraction expansion (a small manual map; libraries like `contractions` do this more fully)
contraction_map = {
    "can't": "cannot", "won't": "will not", "you're": "you are",
    "i'm": "i am", "don't": "do not", "isn't": "is not",
}
expanded = contracted
for contraction, full in contraction_map.items():
    expanded = expanded.replace(contraction, full)
print("expanded:", expanded)

# Repeated characters: "stoooopid" -> often collapsed to "stopid"/"stupid"-ish, but the REPETITION
# itself is a signal of emphasis/intensity -- worth preserving as a separate feature sometimes.
repeated_text = "you are soooo stupiiiid"
collapsed = re.sub(r"(.)\1{2,}", r"\1", repeated_text)   # 3+ repeats -> just 1 character
print("collapsed:", collapsed)

# Alternative: count repeated-character "shouting" as its own feature instead of deleting it
num_elongated_words = len(re.findall(r"(.)\1{2,}", repeated_text))
print("elongated-character count (an intensity signal):", num_elongated_words)

## 7. spaCy — tokenization + lemmatization + POS tagging in one pipeline

spaCy is a more modern, production-oriented NLP library than nltk. One `nlp(text)` call gives
tokens, lemmas, part-of-speech tags, and more, all at once.

In [ ]:
doc = nlp("You are being extremely stupid and worthless right now")

for token in doc:
    print(f"{token.text:<12}{token.lemma_:<12}{token.pos_:<8}is_stop={token.is_stop}")

## 8. Putting it together: a configurable preprocessing function

Deliberately configurable — for toxicity/cyberbullying detection you often want to KEEP some
signals (punctuation intensity, capitalization, emojis) that a generic NLP pipeline would strip.

In [ ]:
def preprocess_text(text, lowercase=True, remove_urls=True, remove_mentions=True,
                     keep_hashtag_word=True, remove_numbers=True, remove_stopwords=False,
                     stem=False):
    if remove_urls:
        text = re.sub(r"http\S+|www\.\S+", "", text)
    if remove_mentions:
        text = re.sub(r"@\w+", "", text)
    if keep_hashtag_word:
        text = re.sub(r"#(\w+)", r"\1", text)
    if remove_numbers:
        text = re.sub(r"\d+", "", text)
    if lowercase:
        text = text.lower()

    tokens = word_tokenize(text)

    if remove_stopwords:
        tokens = [t for t in tokens if t not in stop_words]
    if stem:
        tokens = [stemmer.stem(t) for t in tokens]

    return " ".join(tokens)

sample = "Check this out http://x.com @john #StopBullying123 you're SO stupid!!! 😢"
print("original: ", sample)
print("processed:", preprocess_text(sample))
print("aggressive:", preprocess_text(sample, remove_stopwords=True, stem=True))

## Exercise

In [ ]:
# --- Try it yourself ---
# 1. Add 5 of your own real-looking social-media-style sentences (with hashtags, mentions, emojis)
#    and run them through preprocess_text() with different option combinations.
# 2. Write a small feature extractor that, for a raw text, counts: num_hashtags, num_mentions,
#    num_exclamations, has_elongated_word (reuse the regex from part 6) -- these become numeric
#    features you can add alongside TF-IDF later.
# 3. Compare stemmer.stem() vs lemmatizer.lemmatize() on 10 words from YOUR dataset's vocabulary
#    once you have it -- note any cases where stemming produces a non-word.
# 4. Decide, and write 2-3 sentences justifying: for your cyberbullying paper, will you strip
#    or preserve emojis/punctuation/capitalization? What's your reasoning?

---
### Next up: **Topic 22 — Bag of Words**.

Say "next" when you're ready.